# Analysis for the Friedman benchmark

TODO Update with new SRB samples

The data sets are:

1. `f1a`: $f_1$ on 100 data points, no distractors
2. `f1b`: $f_1$ on 1000 data points, no distractors
3. `f1c`: $f_1$ on 100 data points, 5 columns of distractors
4. `f1d`: $f_1$ on 1000 data points, 5 columns of distractors

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
from sympy.abc import a, b, c, d, e, f, g, x

## Loading data

In [ ]:
f1expr = sympy.sympify("10*sin(π*x1*x2) + 20*(x3 - 1/2)^2 + 10*x4 + 5*x5")
f1expr

In [ ]:
sympy.expand(f1expr)

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(au.parse_if_needed)
full_report["sympy_defuzz"] = full_report.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
full_report.run_set.unique()

Make rows indexable by run set, data set, and sample number.

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
fr2 = full_report.set_index(["run_set", "data_set", "sample_num"])

In [ ]:
all_min_mse_ixs = fr2.groupby(level=["run_set", "data_set"]).mse.idxmin()
fr2.loc[all_min_mse_ixs]

In [ ]:
srb2_key = "SRB-2026-07-13-1130"
srb3_key = "SRB-2026-08-03-1000"
cht1_key = "CHT-2026-08-01-1800"
srb2 = fr2.loc[srb2_key]
srb2s = fr2.loc[srb2_key + "-subset"]
srb3 = fr2.loc[srb3_key]
srb3s = fr2.loc[srb3_key + "-subset"]
cht1 = fr2.loc[cht1_key]
cht1s = fr2.loc[cht1_key + "-subset"]

In [ ]:
srb2.groupby(level=["data_set"]).size()

In [ ]:
srb2s.groupby(level=["data_set"]).size()

In [ ]:
cht1.groupby(level=["data_set"]).size()

In [ ]:
cht1s.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb2_min_mse_ixs = srb2.groupby(level=["data_set"]).mse.idxmin()
srb2s_min_mse_ixs = srb2s.groupby(level=["data_set"]).mse.idxmin()
srb3_min_mse_ixs = srb3.groupby(level=["data_set"]).mse.idxmin()
srb3s_min_mse_ixs = srb3s.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()
cht1s_min_mse_ixs = cht1s.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb2.loc[srb2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2s.loc[srb2s_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb3.loc[srb3_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1.loc[cht1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1s.loc[cht1s_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

## Main SRB and CHT results

In [ ]:
srb_key = srb2_key
srb = srb2
srb_min_mse_ixs = srb2_min_mse_ixs
srbs_key = srb2_key + "-subset"
srbs = srb2s
srbs_min_mse_ixs = srb2s_min_mse_ixs
cht_key = cht1_key
cht = cht1
cht_min_mse_ixs = cht1_min_mse_ixs
chts_key = cht1_key + "-subset"
chts = cht1s
chts_min_mse_ixs = cht1s_min_mse_ixs

In [ ]:
au.mse_threshold_table(fr2)

In [ ]:
srb_threshold_table = au.mse_threshold_table(fr2.loc[[srb_key]])
srb_threshold_table.to_csv("Generated/srb_threshold_table.csv")
srb_threshold_table.to_latex("Generated/srb_threshold_table.tex")
srb_threshold_table

In [ ]:
cht_threshold_table = au.mse_threshold_table(fr2.loc[[cht_key]])
cht_threshold_table.to_csv("Generated/cht_threshold_table.csv")
cht_threshold_table.to_latex("Generated/cht_threshold_table.tex")
cht_threshold_table

In [ ]:
plot_params = {
    "complexity_lims": (0, 449),
    "complexity_col": "complexity_defuzz",
    "mse_lims": (1.0e-12, 0.99e2),
    "complexity_binwidth": 20,
    "mse_binwidth": 0.8,
    #"spiffy_titles": ["f1a", "f1b", "f1c", "f1d"],
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
plot_params_bd = {
    **plot_params,
    #"spiffy_titles": ["f1b", "f1d"]
    }

In [ ]:
fig = au.complexity_mse_displot(srb, file_stem="SRB-complexity-mse-displot", **plot_params)

In [ ]:
au.complexity_mse_displot(srbs, file_stem="srbs-complexity-mse-displot", **plot_params_bd)

In [ ]:
fig = au.complexity_mse_displot(cht, file_stem="cht-complexity-mse-displot", **plot_params)

In [ ]:
au.complexity_mse_displot(chts, file_stem="chts-complexity-mse-displot", **plot_params_bd)

In [ ]:
au.count_by_threshold(srb, threshold=1.0e-7)

In [ ]:
au.count_by_threshold(srbs, threshold=1.0e-7)

A lot of trig cruft.

In [ ]:
srb.loc["f1a"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-5))

In [ ]:
srb.loc["f1a"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1e-4))

There are 22 clearly correct here:

In [ ]:
srb.loc["f1a"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1b"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=1.0e-5))

In [ ]:
srb.loc["f1b"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1c"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1d"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

## Does sub-setting help with the large data set?

In [ ]:
srbs.loc["f1b"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=0.02))

In [ ]:
srbs_f1d = srbs.loc["f1d"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=0.02))
srbs_f1d

Here's a good example of cruft.

In [ ]:
srbs_f1d.loc[26]

In [ ]:
srbs.loc[("f1d", 26), "sympy"]

In [ ]:
print(sympy.latex(srbs_f1d.loc[26]))

In [ ]:
sympy.series(-40.696*sympy.sin(x+1)+40.683, x, 0, 3).evalf()

In [ ]:
sympy.expand(20*(x - 1/2)**2)

## Wide-range data sets

I'm surprised to see this:
Expanding the range of the random $x_j$'s to $[-2,2]$ doesn't improve things.
I was hoping it would cut down on cruft.
It actually makes things worse.
I think the $\sin$ term is harder to find here because the polynomial terms are so much larger.

In [ ]:
srb.loc["f1aw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1bw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1cw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srb.loc["f1dw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srbs.loc["f1bw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

In [ ]:
srbs.loc["f1dw"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=0.02))

## Larger $\lambda_b$

I tried everything again with $\lambda_b = 10^{-10}$ to see what effect it would have on fuzz.
These are the `srb3` trials.

In [ ]:
srb3.loc["f1a"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))

In [ ]:
srb3.loc["f1b"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))

In [ ]:
srb3.loc["f1c"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))

In [ ]:
srb3.loc["f1d"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))

In [ ]:
srb3s.loc["f1b"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))

In [ ]:
srb3s.loc["f1d"].iloc[0:10].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e).evalf(), tolerance=5e-4))